# CS156: Pipeline - Second Draft

**Can You Diagnose Autism From Brain Noise?**

There’s a persistent idea in computational neuroscience that you can look at a brain scan and read *something* meaningful: a diagnosis, a trait, or a difference in how someone thinks. It’s a compelling idea. It’s also, at best, an approximation.

This project starts from a deliberately uncomfortable question:

> If you take a noisy, indirect measurement of brain activity and reduce it to correlations between ~200 regions, is there actually enough signal left to distinguish individuals with Autism Spectrum Disorder (ASD) from controls?

The data used here are resting-state fMRI time series, summarized as functional connectivity matrices. In practice, this means:
- we are not observing neural activity directly  
- we are not measuring interactions between regions  
- we are measuring statistical co-variation under a long chain of preprocessing assumptions  

So the object we feed into our models is already several steps removed from anything we might confidently call “brain function.”

Given that, the goal of this project is not to build a diagnostic tool, but rather to test how different modeling assumptions behave when applied to this representation.

Each subject is represented as a connectivity matrix derived from ROI time series (`rois_cc200`). From there, we ask a simple but pointed question:

> Does treating the brain as a graph actually help, or are we just adding structure to noise?

To answer this, we compare four approaches:

* Logistic Regression on flattened connectivity features  
* A multilayer perceptron (MLP) on the same representation  
* A graph convolutional network (GCN) that assumes connectivity structure matters  
* A graph attention network (GAT) that learns which connections to emphasize  

All models operate on the same underlying data, allowing differences in performance to reflect model assumptions rather than differences in input.

If graph-based models outperform simpler baselines, that suggests there is meaningful relational structure in the data. If they do not, it raises a more uncomfortable possibility: that increasing model sophistication does not recover signal that may not be there to begin with.

The notebook proceeds as follows:

1. Load and verify the curated dataset  
2. Construct functional connectivity matrices  
3. Prepare inputs for each modeling approach  
4. Train and evaluate models under consistent conditions  
5. Compare results and analyze differences  

The aim is not to settle whether functional connectivity can diagnose ASD, but to probe how far this representation can be pushed—and where it breaks down.

# Data

## Data Source

## Preprocessing Done on Original Data

# Loading the Data and Feature Engineering

In [4]:
# loading data and building correlation matrices (verbose debug)
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

base_dir = Path("data") / "abide_fmri"
ts_dir = base_dir / "timeseries"
fc_dir = base_dir / "connectivity_matrices"

print("[INIT] Ensuring output directory exists...")
fc_dir.mkdir(parents=True, exist_ok=True)

subjects_path = base_dir / "subjects_clean.csv"

# --- get subject list ---
print("[STEP 1] Getting subject list...")

if subjects_path.exists():
    print("  -> Reading CSV")
    subjects_df = pd.read_csv(subjects_path)
    print("  -> CSV loaded")

    file_ids = subjects_df["FILE_ID"].dropna().astype(str).drop_duplicates().tolist()
    print("  -> Extracted FILE_IDs")
else:
    print("  -> No CSV found, scanning timeseries folder")
    file_ids = sorted(p.stem for p in ts_dir.glob("*.1D"))

print(f"[STEP 1 DONE] {len(file_ids)} subjects")

valid_ids = []
failed_ids = []
skipped_ids = []
roi_count = None

print("[STEP 2] Starting processing loop...")

# --- main loop ---
for i, fid in enumerate(tqdm(file_ids, desc="Processing")):
    ts_path = ts_dir / f"{fid}.1D"
    fc_path = fc_dir / f"{fid}.npy"

    # skip if already processed
    if fc_path.exists():
        skipped_ids.append(fid)
        valid_ids.append(fid)
        continue

    if not ts_path.exists():
        print(f"[WARN] Missing file: {fid}")
        failed_ids.append(fid)
        continue

    try:
        ts = np.loadtxt(ts_path)

        if ts.ndim != 2:
            print(f"[WARN] Bad shape (ndim) for {fid}: {ts.ndim}")
            failed_ids.append(fid)
            continue

        # enforce 200 ROI format
        if ts.shape[1] == 200:
            pass
        elif ts.shape[0] == 200:
            ts = ts.T
            print(f"[INFO] Transposed: {fid}")
        else:
            print(f"[WARN] Unexpected shape for {fid}: {ts.shape}")
            failed_ids.append(fid)
            continue

        corr = np.corrcoef(ts, rowvar=False)
        corr = np.nan_to_num(corr).astype(np.float32)
        np.fill_diagonal(corr, 0)

        if roi_count is None:
            roi_count = corr.shape[0]
            print(f"[INFO] ROI count set to {roi_count}")

        if corr.shape != (roi_count, roi_count):
            print(f"[WARN] Shape mismatch for {fid}: {corr.shape}")
            failed_ids.append(fid)
            continue

        np.save(fc_path, corr)
        valid_ids.append(fid)

    except Exception as e:
        print(f"[ERROR] Failed {fid}: {e}")
        failed_ids.append(fid)

print("[STEP 2 DONE] Processing finished")

# --- build index ---
print("[STEP 3] Building index...")

index_df = pd.DataFrame({"FILE_ID": valid_ids})

if subjects_path.exists():
    subjects_clean = subjects_df[subjects_df["FILE_ID"].isin(valid_ids)].copy()
    subjects_clean.to_csv(base_dir / "subjects_with_fc.csv", index=False)

    index_df = index_df.merge(
        subjects_clean[["FILE_ID", "DX_GROUP"]],
        on="FILE_ID",
        how="left"
    )

index_df.to_csv(base_dir / "connectivity_index.csv", index=False)

# --- summary ---
print("\n--- Summary ---")
print(f"Processed new: {len(valid_ids) - len(skipped_ids)}")
print(f"Skipped: {len(skipped_ids)}")
print(f"Failed: {len(failed_ids)}")

if roi_count is not None:
    print(f"Matrix shape: ({roi_count}, {roi_count})")

if failed_ids:
    print("Example failures:", failed_ids[:10])

[INIT] Ensuring output directory exists...
[STEP 1] Getting subject list...
  -> Reading CSV
  -> CSV loaded
  -> Extracted FILE_IDs
[STEP 1 DONE] 1035 subjects
[STEP 2] Starting processing loop...


Processing:   2%|▏         | 22/1035 [00:00<00:04, 211.87it/s]

[INFO] ROI count set to 200


/Users/suiseinakagawa/Desktop/University/Academics/3rd year/Spring 2026/CS156/assignments/abide-graph-classifier/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/suiseinakagawa/Desktop/University/Academics/3rd year/Spring 2026/CS156/assignments/abide-graph-classifier/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Processing: 100%|██████████| 1035/1035 [00:04<00:00, 241.12it/s]

[STEP 2 DONE] Processing finished
[STEP 3] Building index...

--- Summary ---
Processed new: 1035
Skipped: 0
Failed: 0
Matrix shape: (200, 200)


In [7]:
# Build sparse PyG graphs from connectivity matrices (top-k edges per node)
# Build PyG graphs (improved: meaningful node features + cleaner edges)

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch_geometric.data import Data

base_dir = Path("data") / "abide_fmri"
fc_dir = base_dir / "connectivity_matrices"
index_path = base_dir / "connectivity_index.csv"
pyg_dir = base_dir / "pyg"
pyg_dir.mkdir(parents=True, exist_ok=True)

TOP_K = 10
USE_ABS = False  # <- set True if you already committed to abs correlations


def matrix_to_topk_edge_index(adj: np.ndarray, k: int = 10):
    """Top-k edges per node, symmetric, no self-loops."""
    n = adj.shape[0]
    a = adj.copy().astype(np.float32)
    np.fill_diagonal(a, 0.0)

    edges = set()

    for i in range(n):
        row = a[i]

        # get indices of top-k strongest connections (by magnitude)
        idx = np.argsort(np.abs(row))[-k:]

        for j in idx:
            if i == j:
                continue
            u, v = sorted((i, j))
            edges.add((u, v))

    if not edges:
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float32)

    edge_list = []
    weights = []

    for (u, v) in edges:
        w = a[u, v]
        edge_list.append([u, v])
        edge_list.append([v, u])
        weights.append(w)
        weights.append(w)

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(weights, dtype=torch.float32)

    return edge_index, edge_weight


# --- load index ---
index_df = pd.read_csv(index_path)
label_map = {1: 1, 2: 0}

graphs = []
skipped = []

for _, row in index_df.iterrows():
    fid = str(row["FILE_ID"])
    y_raw = row.get("DX_GROUP", np.nan)

    mat_path = fc_dir / f"{fid}.npy"
    if not mat_path.exists():
        skipped.append(fid)
        continue

    adj = np.load(mat_path)

    if adj.ndim != 2 or adj.shape[0] != adj.shape[1]:
        skipped.append(fid)
        continue

    if USE_ABS:
        adj = np.abs(adj)

    # --- edges ---
    edge_index, edge_weight = matrix_to_topk_edge_index(adj, k=TOP_K)

    # --- node features (IMPORTANT CHANGE) ---
    # each node = its connectivity profile
    x = torch.tensor(adj, dtype=torch.float32)
    x = (x - x.mean()) / (x.std() + 1e-6)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_weight,
        y=None,
        file_id=fid
    )

    if pd.notna(y_raw):
        data.y = torch.tensor([label_map[int(y_raw)]], dtype=torch.long)

    graphs.append(data)

# --- save ---
out_path = pyg_dir / f"graphs_top{TOP_K}.pt"
torch.save(graphs, out_path)

print(f"Saved {len(graphs)} graphs → {out_path}")
print(f"Skipped: {len(skipped)}")

if graphs:
    g = graphs[0]
    print(f"Example graph: nodes={g.num_nodes}, edges={g.edge_index.shape[1]}, features={g.x.shape}")

Saved 1035 graphs → data/abide_fmri/pyg/graphs_top10.pt
Skipped: 0
Example graph: nodes=200, edges=2660, features=torch.Size([200, 200])


# Exploratory Data Analysis

# Splitting the Data

# Model Selection

## Baseline: Predicting the Majority Class

## Logistic Regression 

## Multilayer Perceptron (MLP) 

## Graph Convolutional Network (GCN)  

### Mathematical Basis of GCNs

## Graph Attention Network (GAT)

### Mathematical Basis of GATs

# Training with Cross-Validation and Hyperparameter Tuning

# Evaluation on the Test Set

# Results and Conclusions 

# Limitations 

# Executive Summary

# AI Statement

# References